# Lab de Limpeza - Camada Silver: Financeiro

In [55]:
import pandas as pd
import numpy as np
from pathlib import Path
import glob

In [23]:
if "__file__" in globals():
    BASE_DIR = Path(__file__).resolve().parents[3]
else:
    BASE_DIR = Path.cwd().resolve().parents[2]

In [24]:
BRONZE_PATH = BASE_DIR / "data" / "bronze" / "financeiro"
SILVER_PATH = BASE_DIR / "data" / "silver" / "financeiro"

In [25]:
files = glob.glob(str(BRONZE_PATH / "**/*.csv"), recursive=True)

In [26]:
print("BASE_DIR:", BASE_DIR)
print("BRONZE_PATH:", BRONZE_PATH)
print("Arquivos encontrados:", files)

BASE_DIR: /home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case
BRONZE_PATH: /home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/financeiro
Arquivos encontrados: ['/home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/financeiro/ingestion_date=2026-04-27T12:01:49/financeiro.csv', '/home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/financeiro/ingestion_date=2026-04-27T17:45:52/financeiro.csv', '/home/elladarte/Documentos/teste-tecnico-uberlandia-refresco/refresco-data-case/data/bronze/financeiro/ingestion_date=2026-04-27T17:49:03/financeiro.csv']


In [27]:
df = pd.concat(
    [pd.read_csv(file, encoding="utf-8", sep=",") for file in files],
    ignore_index=True
)
df

,id_financeiro,id_venda,tipo_lancamento,valor_bruto,valor_imposto,valor_liquido,competencia,status,_ingested_at,_source_file
0,F0001614,V002457,AJUSTE,3245.87,389.50,2856.37,202407,PENDENTE,2026-04-27 12:01:49.760802,financeiro.csv
1,F0001473,V001995,VENDA,4023.86,482.86,3541.00,202408,CANCELADO,2026-04-27 12:01:49.760802,financeiro.csv
2,F0000317,V003128,AJUSTE,501.30,60.16,441.14,202412,PAGO,2026-04-27 12:01:49.760802,financeiro.csv
3,F0000604,V002963,AJUSTE,1514.17,181.70,1332.47,202412,PENDENTE,2026-04-27 12:01:49.760802,financeiro.csv
4,F0000669,V001578,DEVOLUCAO,1362.83,163.54,1199.29,202404,PENDENTE,2026-04-27 12:01:49.760802,financeiro.csv
...,...,...,...,...,...,...,...,...,...,...
7387,F0000090,V001649,DEVOLUCAO,3666.09,439.93,3226.16,202410,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv
7388,F0002024,V001451,VENDA,1876.41,225.17,1651.24,202404,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv
7389,F0001653,V002514,BONIFICACAO,1312.09,157.45,1154.64,202406,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv
7390,F0002317,V000403,BONIFICACAO,3858.07,462.97,3395.10,202408,PAGO,2026-04-27 17:49:03.897623,financeiro.csv


Tipagem

In [28]:
df["_ingested_at"] = pd.to_datetime(df["_ingested_at"], errors="coerce")

cols_float = ["valor_bruto", "valor_imposto", "valor_liquido"]

for col in cols_float:
    df[col] = pd.to_numeric(df[col], errors="coerce")

Padronização de strings

In [29]:
cols_str = ["tipo_lancamento", "status"]

for col in cols_str:
    df[col] = df[col].astype(str).str.strip().str.upper()

Tratamento da competência

In [30]:
df["competencia"].unique()

array(['202407', '202408', '202412', '202404', '202402', '202411',
       '202405', '202400', '202409', '202406', '202403', '202401',
       '202410', '2024/01', '2024/03', '2024/12', '2024/10', '2024/04',
       '2024/07', '2024/02', '2024/05', '2024/08', '2024/11', '2024/06',
       '2024/09'], dtype=object)

In [31]:
df["competencia_raw"] = df["competencia"]

df["competencia"] = (
    df["competencia"]
    .astype(str)
    .str.replace("/", "", regex=False)
    .str.strip()
)

In [32]:
df

,id_financeiro,id_venda,tipo_lancamento,valor_bruto,valor_imposto,valor_liquido,competencia,status,_ingested_at,_source_file,competencia_raw
0,F0001614,V002457,AJUSTE,3245.87,389.50,2856.37,202407,PENDENTE,2026-04-27 12:01:49.760802,financeiro.csv,202407
1,F0001473,V001995,VENDA,4023.86,482.86,3541.00,202408,CANCELADO,2026-04-27 12:01:49.760802,financeiro.csv,202408
2,F0000317,V003128,AJUSTE,501.30,60.16,441.14,202412,PAGO,2026-04-27 12:01:49.760802,financeiro.csv,202412
3,F0000604,V002963,AJUSTE,1514.17,181.70,1332.47,202412,PENDENTE,2026-04-27 12:01:49.760802,financeiro.csv,202412
4,F0000669,V001578,DEVOLUCAO,1362.83,163.54,1199.29,202404,PENDENTE,2026-04-27 12:01:49.760802,financeiro.csv,202404
...,...,...,...,...,...,...,...,...,...,...,...
7387,F0000090,V001649,DEVOLUCAO,3666.09,439.93,3226.16,202410,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,202410
7388,F0002024,V001451,VENDA,1876.41,225.17,1651.24,202404,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,202404
7389,F0001653,V002514,BONIFICACAO,1312.09,157.45,1154.64,202406,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,202406
7390,F0002317,V000403,BONIFICACAO,3858.07,462.97,3395.10,202408,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,202408


In [46]:
df["tipo_lancamento"].unique()

array(['DEVOLUCAO', 'AJUSTE', 'BONIFICACAO', 'VENDA', 'OUTRO', 'V',
       'BONIF.', 'D', 'DEV', 'VND', 'BONIFICAçãO', 'DEVOLUçãO', 'BONIF',
       'DEVOL', 'VENDAS'], dtype=object)

In [34]:
df["tipo_lancamento_raw"] = df["tipo_lancamento"]

Corrigir encoding

In [35]:
def fix_encoding(text):
    try:
        return text.encode("latin1").decode("utf-8")
    except:
        return text

df["tipo_lancamento"] = (
    df["tipo_lancamento"]
    .astype(str)
    .apply(fix_encoding)
)

Normalizar

In [36]:
df["tipo_lancamento_norm"] = (
    df["tipo_lancamento"]
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

mapeamento semantico

In [37]:
map_tipo = {
    # VENDA
    "VENDA": "VENDA",
    "VENDAS": "VENDA",
    "VND": "VENDA",
    "V": "VENDA",

    # DEVOLUÇÃO
    "DEVOLUCAO": "DEVOLUCAO",
    "DEV": "DEVOLUCAO",
    "DEVOL": "DEVOLUCAO",
    "D": "DEVOLUCAO",
    "DEVOLUÇÃO": "DEVOLUCAO",

    # BONIFICAÇÃO
    "BONIFICACAO": "BONIFICACAO",
    "BONIF": "BONIFICACAO",
    "BONIFICACAO": "BONIFICACAO",
    "BONIFICAÇÃO": "BONIFICACAO",

    # OUTROS
    "AJUSTE": "AJUSTE",
    "OUTRO": "OUTRO",
}

Aplicar e marcar desconhecidos

In [38]:
df["tipo_lancamento_padrao"] = df["tipo_lancamento_norm"].map(map_tipo)

df["tipo_lancamento_desconhecido"] = df["tipo_lancamento_padrao"].isna()

In [39]:
df[df["tipo_lancamento_desconhecido"]]["tipo_lancamento_raw"].value_counts()

Series([], Name: count, dtype: int64)

In [47]:
df["tipo_lancamento_padrao"].unique()

array(['DEVOLUCAO', 'AJUSTE', 'BONIFICACAO', 'VENDA', 'OUTRO'],
      dtype=object)

Status

In [49]:
df["status"].unique()

array(['CANCELADO', 'PENDENTE', 'PAGO', 'NAN'], dtype=object)

In [52]:
df["status_raw"] = df["status"]

In [50]:
df["competencia_dt"] = pd.to_datetime(
    df["competencia"],
    format="%Y%m",
    errors="coerce"
)

In [53]:
df["status_norm"] = (
    df["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [56]:
df["status_norm"] = df["status_norm"].replace("NAN", np.nan)

In [57]:
map_status = {
    "PAGO": "PAGO",
    "PENDENTE": "PENDENTE",
    "CANCELADO": "CANCELADO"
}

df["status_padrao"] = df["status_norm"].map(map_status)

In [58]:
df["status_desconhecido"] = df["status_padrao"].isna() & df["status_norm"].notna()
df["status_nulo"] = df["status_norm"].isna()

In [59]:
df["status_padrao"].value_counts(dropna=False)

status_padrao
PAGO         854
PENDENTE     832
CANCELADO    768
NaN           10
Name: count, dtype: int64

In [60]:
df[df["status_desconhecido"]]["status_raw"].value_counts()

Series([], Name: count, dtype: int64)

In [51]:
df

,id_financeiro,id_venda,tipo_lancamento,valor_bruto,valor_imposto,valor_liquido,competencia,status,_ingested_at,_source_file,competencia_raw,tipo_lancamento_raw,tipo_lancamento_norm,tipo_lancamento_padrao,tipo_lancamento_desconhecido,competencia_dt,competencia_invalida
6565,F0000667,V000980,DEVOLUCAO,1236.73,148.41,1088.32,2024-09-01,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,202409,DEVOLUCAO,DEVOLUCAO,DEVOLUCAO,False,2024-09-01,False
6566,F0001176,V000514,AJUSTE,4648.93,557.87,4091.06,2024-01-01,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,202401,AJUSTE,AJUSTE,AJUSTE,False,2024-01-01,False
6567,F0002113,V001602,BONIFICACAO,4992.26,599.07,4393.19,2024-07-01,PENDENTE,2026-04-27 17:49:03.897623,financeiro.csv,202407,BONIFICACAO,BONIFICACAO,BONIFICACAO,False,2024-07-01,False
6568,F0002154,V000332,DEVOLUCAO,2498.57,299.83,2198.74,2024-01-01,PENDENTE,2026-04-27 17:49:03.897623,financeiro.csv,202401,DEVOLUCAO,DEVOLUCAO,DEVOLUCAO,False,2024-01-01,False
6569,F0001659,V002551,AJUSTE,906.95,108.83,798.12,2024-04-01,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,202404,AJUSTE,AJUSTE,AJUSTE,False,2024-04-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5750,F0001850,V002202,AJUSTE,4829.50,579.54,4249.96,2024-10-01,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,202410,AJUSTE,AJUSTE,AJUSTE,False,2024-10-01,False
5751,F0000829,V000707,AJUSTE,315.75,37.89,277.86,2024-12-01,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,202412,AJUSTE,AJUSTE,AJUSTE,False,2024-12-01,False
5752,F0001882,V002158,DEVOLUCAO,737.22,88.47,648.75,2024-12-01,PENDENTE,2026-04-27 17:49:03.897623,financeiro.csv,202412,DEVOLUCAO,DEVOLUCAO,DEVOLUCAO,False,2024-12-01,False
5745,F0001801,V002587,BONIFICACAO,2603.39,312.41,2290.98,2024-11-01,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,202411,BONIFICACAO,BONIFICACAO,BONIFICACAO,False,2024-11-01,False


Detectar inválidos

In [42]:
df["competencia_invalida"] = df["competencia_dt"].isna()

In [43]:
df[df["competencia_invalida"]]["competencia_raw"].value_counts()

competencia_raw
202400    30
Name: count, dtype: int64

In [44]:
df["competencia"] = df["competencia_dt"]

In [45]:
df = df.sort_values("_ingested_at")

df = df.drop_duplicates(
    subset=["id_financeiro"],
    keep="last"
)

In [62]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2464 entries, 6565 to 7391
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   id_financeiro                 2464 non-null   object        
 1   id_venda                      2454 non-null   object        
 2   tipo_lancamento               2464 non-null   object        
 3   valor_bruto                   2464 non-null   float64       
 4   valor_imposto                 2464 non-null   float64       
 5   valor_liquido                 2464 non-null   float64       
 6   competencia                   2454 non-null   datetime64[ns]
 7   status                        2464 non-null   object        
 8   _ingested_at                  2464 non-null   datetime64[ns]
 9   _source_file                  2464 non-null   object        
 10  competencia_raw               2464 non-null   object        
 11  tipo_lancamento_raw           24

In [ ]:
df[["id_financeiro","id_venda","valor_bruto","valor_imposto","valor_liquido","status","_ingested_at","_source_file",]]

,id_financeiro,id_venda,tipo_lancamento,valor_bruto,valor_imposto,valor_liquido,competencia,status,_ingested_at,_source_file,...,tipo_lancamento_norm,tipo_lancamento_padrao,tipo_lancamento_desconhecido,competencia_dt,competencia_invalida,status_raw,status_norm,status_padrao,status_desconhecido,status_nulo
6565,F0000667,V000980,DEVOLUCAO,1236.73,148.41,1088.32,2024-09-01,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,...,DEVOLUCAO,DEVOLUCAO,False,2024-09-01,False,CANCELADO,CANCELADO,CANCELADO,False,False
6566,F0001176,V000514,AJUSTE,4648.93,557.87,4091.06,2024-01-01,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,...,AJUSTE,AJUSTE,False,2024-01-01,False,CANCELADO,CANCELADO,CANCELADO,False,False
6567,F0002113,V001602,BONIFICACAO,4992.26,599.07,4393.19,2024-07-01,PENDENTE,2026-04-27 17:49:03.897623,financeiro.csv,...,BONIFICACAO,BONIFICACAO,False,2024-07-01,False,PENDENTE,PENDENTE,PENDENTE,False,False
6568,F0002154,V000332,DEVOLUCAO,2498.57,299.83,2198.74,2024-01-01,PENDENTE,2026-04-27 17:49:03.897623,financeiro.csv,...,DEVOLUCAO,DEVOLUCAO,False,2024-01-01,False,PENDENTE,PENDENTE,PENDENTE,False,False
6569,F0001659,V002551,AJUSTE,906.95,108.83,798.12,2024-04-01,CANCELADO,2026-04-27 17:49:03.897623,financeiro.csv,...,AJUSTE,AJUSTE,False,2024-04-01,False,CANCELADO,CANCELADO,CANCELADO,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5750,F0001850,V002202,AJUSTE,4829.50,579.54,4249.96,2024-10-01,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,...,AJUSTE,AJUSTE,False,2024-10-01,False,PAGO,PAGO,PAGO,False,False
5751,F0000829,V000707,AJUSTE,315.75,37.89,277.86,2024-12-01,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,...,AJUSTE,AJUSTE,False,2024-12-01,False,PAGO,PAGO,PAGO,False,False
5752,F0001882,V002158,DEVOLUCAO,737.22,88.47,648.75,2024-12-01,PENDENTE,2026-04-27 17:49:03.897623,financeiro.csv,...,DEVOLUCAO,DEVOLUCAO,False,2024-12-01,False,PENDENTE,PENDENTE,PENDENTE,False,False
5745,F0001801,V002587,BONIFICACAO,2603.39,312.41,2290.98,2024-11-01,PAGO,2026-04-27 17:49:03.897623,financeiro.csv,...,BONIFICACAO,BONIFICACAO,False,2024-11-01,False,PAGO,PAGO,PAGO,False,False
